# 07 — Inference assets and report-ready outputs

This notebook prepared a small inference sample, model-card-style metadata, and report-ready tables for the Streamlit app.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from project_package.config import ensure_project_dirs, PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, TARGET
from project_package.plotting import save_figure
from project_package.reporting import save_table

ensure_project_dirs()
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

from project_package.models import load_model, predict_proba_positive, save_model
from project_package.inference import create_adverse_action_reasons

In [2]:
metadata_path = TABLES_DIR / "06_best_model_metadata.csv"
if not metadata_path.exists():
    raise FileNotFoundError("Run Notebook 06 before Notebook 07.")
metadata = pd.read_csv(metadata_path)
metadata

,best_model_name,best_probability_column,selection_metric,optimal_threshold_by_cost
0,xgboost,proba_xgboost,pr_auc,0.37


## Inference sample

A small sample of feature rows was saved without the target. The Streamlit app used this table to demonstrate model scoring.

In [3]:
test_df = pd.read_csv(PROCESSED_DIR / "test_split.csv")
inference_sample = test_df.drop(columns=[TARGET]).sample(n=min(200, len(test_df)), random_state=42)
inference_sample.to_csv(PROCESSED_DIR / "inference_sample.csv", index=False)
print(f"Saved inference sample with shape {inference_sample.shape}")
inference_sample.head()

Saved inference sample with shape (200, 48)


,credit_limit,sex,education,marriage,age,pay_status_sep,pay_status_aug,pay_status_jul,pay_status_jun,pay_status_may,pay_status_apr,bill_amt_sep,bill_amt_aug,bill_amt_jul,bill_amt_jun,bill_amt_may,bill_amt_apr,pay_amt_sep,pay_amt_aug,pay_amt_jul,pay_amt_jun,pay_amt_may,pay_amt_apr,proxy_recent_repayment_status,proxy_max_delinquency_6m,proxy_mean_delinquency_6m,proxy_delinquency_count_6m,proxy_severe_delinquency_count_6m,proxy_on_time_count_6m,proxy_total_bill_6m,proxy_total_payment_6m,proxy_total_payment_to_bill_6m,ratio_pay_amt_apr_to_bill_amt_apr,ratio_pay_amt_may_to_bill_amt_may,ratio_pay_amt_jun_to_bill_amt_jun,ratio_pay_amt_jul_to_bill_amt_jul,ratio_pay_amt_aug_to_bill_amt_aug,ratio_pay_amt_sep_to_bill_amt_sep,proxy_payment_ratio_mean_6m,proxy_payment_ratio_volatility_6m,proxy_payment_consistency_6m,proxy_bill_volatility_6m,proxy_payment_volatility_6m,proxy_utilization_mean_6m,proxy_utilization_max_6m,proxy_recent_payment_to_limit,proxy_recent_bill_to_limit,proxy_age_to_credit_limit
1782,260000,male,graduate_school,married,37,0,0,3,2,2,2,163553,192724,188441,191134,193673,197104,32000,0,7100,7000,6500,7000,0,3,1.500000,4,4,2,1126629,59600,0.052901,0.035514,0.033562,0.036624,0.037678,0.000000,0.195655,0.056505,0.069674,0.833333,12204.432748,11159.151700,0.722198,0.758092,0.123077,0.629050,0.000142
3917,80000,male,high_school,single,44,2,0,0,0,-2,-2,78950,79788,70319,0,0,0,2815,1858,0,0,0,0,2,2,-0.333333,1,1,5,229057,4673,0.020401,0.000000,0.000000,0.000000,0.000000,0.023287,0.035655,0.009824,0.015713,0.333333,41951.092946,1243.937364,0.477202,0.997350,0.035187,0.986875,0.000550
221,360000,female,graduate_school,single,30,-2,-2,-2,-2,-2,-2,750,1166,398,0,418,1840,1166,398,0,418,1840,0,-2,-2,-2.000000,0,0,6,4572,3822,0.835958,0.000000,4.401914,0.000000,0.000000,0.341338,1.554667,1.049653,1.749269,0.666667,656.798295,727.140977,0.002117,0.005111,0.003239,0.002083,0.000083
2135,90000,female,university,married,26,2,0,0,0,0,0,89947,91929,88433,88010,87814,87831,5000,4300,3500,3500,3700,3100,2,2,0.333333,1,1,5,533964,23100,0.043261,0.035295,0.042135,0.039768,0.039578,0.046775,0.055588,0.043190,0.007138,1.000000,1646.230847,686.294397,0.988822,1.021433,0.055556,0.999411,0.000289
5224,50000,male,university,single,25,-1,-1,-1,-1,0,0,5814,-73,11427,191,7391,6554,0,11500,191,7200,2000,7000,-1,0,-0.666667,0,0,6,31377,27891,0.888900,1.068050,0.270599,5.000000,0.016715,0.000000,0.000000,1.059227,1.973953,0.833333,4442.502208,4636.273665,0.104590,0.228540,0.000000,0.116280,0.000500


## Report-ready model card

The model card summarized intended use, non-use cases, data limitations, and governance checks. It was saved as a table so it could be copied into a final report.

In [4]:
model_card = pd.DataFrame([
    {"section": "intended_use", "content": "Research and portfolio demonstration of default-risk scoring using public repayment data and alternative-data-inspired features."},
    {"section": "not_intended_use", "content": "Production lending, automated rejection, or real borrower scoring without consent, governance, regulatory review, and true out-of-time validation."},
    {"section": "data_limitation", "content": "The runnable dataset was not genuine mobile-money data; it was an open credit-default proxy dataset."},
    {"section": "privacy", "content": "True USSD, airtime, and mobile-money records would require strict minimization, consent, encryption, access control, and retention limits."},
    {"section": "fairness", "content": "Group diagnostics were performed only for available proxy groups and did not constitute a complete fairness audit."},
    {"section": "monitoring", "content": "A deployed model would need drift monitoring, calibration monitoring, adverse-action logging, and periodic revalidation."},
])
save_table(model_card, "07_model_card.csv")
model_card

,section,content
0,intended_use,Research and portfolio demonstration of defaul...
1,not_intended_use,"Production lending, automated rejection, or re..."
2,data_limitation,The runnable dataset was not genuine mobile-mo...
3,privacy,"True USSD, airtime, and mobile-money records w..."
4,fairness,Group diagnostics were performed only for avai...
5,monitoring,"A deployed model would need drift monitoring, ..."


## Scoring the inference sample when a scikit-learn best model exists

The saved best pipeline was used to produce a report-ready sample prediction table. If the MLP had won model selection, the app could be adjusted to load the PyTorch model separately.

In [5]:
best_model_path = MODELS_DIR / "best_credit_scoring_pipeline.joblib"
if best_model_path.exists():
    model = load_model("best_credit_scoring_pipeline.joblib")
    sample_scores = predict_proba_positive(model, inference_sample)
    scored_sample = inference_sample.copy()
    scored_sample["predicted_default_probability"] = sample_scores
    scored_sample["risk_band"] = pd.cut(sample_scores, bins=[-0.001, 0.10, 0.25, 0.50, 1.0], labels=["very_low", "low", "medium", "high"])
    scored_sample["illustrative_reasons"] = scored_sample.apply(lambda row: "; ".join(create_adverse_action_reasons(row)), axis=1)
    scored_sample.to_csv(PROCESSED_DIR / "scored_inference_sample.csv", index=False)
    save_table(scored_sample[["predicted_default_probability", "risk_band", "illustrative_reasons"]].head(50), "07_scored_sample_preview.csv")
    display(scored_sample.head())
else:
    print("No scikit-learn best model artifact found. The MLP may have been selected; adjust the app loader if needed.")

,credit_limit,sex,education,marriage,age,pay_status_sep,pay_status_aug,pay_status_jul,pay_status_jun,pay_status_may,pay_status_apr,bill_amt_sep,bill_amt_aug,bill_amt_jul,bill_amt_jun,bill_amt_may,bill_amt_apr,pay_amt_sep,pay_amt_aug,pay_amt_jul,pay_amt_jun,pay_amt_may,pay_amt_apr,proxy_recent_repayment_status,proxy_max_delinquency_6m,proxy_mean_delinquency_6m,proxy_delinquency_count_6m,proxy_severe_delinquency_count_6m,proxy_on_time_count_6m,proxy_total_bill_6m,proxy_total_payment_6m,proxy_total_payment_to_bill_6m,ratio_pay_amt_apr_to_bill_amt_apr,ratio_pay_amt_may_to_bill_amt_may,ratio_pay_amt_jun_to_bill_amt_jun,ratio_pay_amt_jul_to_bill_amt_jul,ratio_pay_amt_aug_to_bill_amt_aug,ratio_pay_amt_sep_to_bill_amt_sep,proxy_payment_ratio_mean_6m,proxy_payment_ratio_volatility_6m,proxy_payment_consistency_6m,proxy_bill_volatility_6m,proxy_payment_volatility_6m,proxy_utilization_mean_6m,proxy_utilization_max_6m,proxy_recent_payment_to_limit,proxy_recent_bill_to_limit,proxy_age_to_credit_limit,predicted_default_probability,risk_band,illustrative_reasons
1782,260000,male,graduate_school,married,37,0,0,3,2,2,2,163553,192724,188441,191134,193673,197104,32000,0,7100,7000,6500,7000,0,3,1.500000,4,4,2,1126629,59600,0.052901,0.035514,0.033562,0.036624,0.037678,0.000000,0.195655,0.056505,0.069674,0.833333,12204.432748,11159.151700,0.722198,0.758092,0.123077,0.629050,0.000142,0.679993,high,recent repayment delays were frequent; recent ...
3917,80000,male,high_school,single,44,2,0,0,0,-2,-2,78950,79788,70319,0,0,0,2815,1858,0,0,0,0,2,2,-0.333333,1,1,5,229057,4673,0.020401,0.000000,0.000000,0.000000,0.000000,0.023287,0.035655,0.009824,0.015713,0.333333,41951.092946,1243.937364,0.477202,0.997350,0.035187,0.986875,0.000550,0.881040,high,recent payments were low relative to outstandi...
221,360000,female,graduate_school,single,30,-2,-2,-2,-2,-2,-2,750,1166,398,0,418,1840,1166,398,0,418,1840,0,-2,-2,-2.000000,0,0,6,4572,3822,0.835958,0.000000,4.401914,0.000000,0.000000,0.341338,1.554667,1.049653,1.749269,0.666667,656.798295,727.140977,0.002117,0.005111,0.003239,0.002083,0.000083,0.411059,medium,
2135,90000,female,university,married,26,2,0,0,0,0,0,89947,91929,88433,88010,87814,87831,5000,4300,3500,3500,3700,3100,2,2,0.333333,1,1,5,533964,23100,0.043261,0.035295,0.042135,0.039768,0.039578,0.046775,0.055588,0.043190,0.007138,1.000000,1646.230847,686.294397,0.988822,1.021433,0.055556,0.999411,0.000289,0.827062,high,recent payments were low relative to outstandi...
5224,50000,male,university,single,25,-1,-1,-1,-1,0,0,5814,-73,11427,191,7391,6554,0,11500,191,7200,2000,7000,-1,0,-0.666667,0,0,6,31377,27891,0.888900,1.068050,0.270599,5.000000,0.016715,0.000000,0.000000,1.059227,1.973953,0.833333,4442.502208,4636.273665,0.104590,0.228540,0.000000,0.116280,0.000500,0.299997,medium,


## Final project checklist

The table below recorded the expected outputs. Missing outputs indicated which notebook needed to be rerun.

In [6]:
checklist_items = [
    ("processed_base_dataset", PROCESSED_DIR / "credit_default_base.csv"),
    ("engineered_feature_dataset", PROCESSED_DIR / "credit_default_features.csv"),
    ("final_model_comparison", TABLES_DIR / "06_final_model_comparison.csv"),
    ("best_model_metadata", TABLES_DIR / "06_best_model_metadata.csv"),
    ("inference_sample", PROCESSED_DIR / "inference_sample.csv"),
    ("streamlit_model_artifact", MODELS_DIR / "best_credit_scoring_pipeline.joblib"),
]
checklist = pd.DataFrame([{"artifact": name, "path": str(path.relative_to(ROOT)), "exists": path.exists()} for name, path in checklist_items])
save_table(checklist, "07_final_artifact_checklist.csv")
checklist

,artifact,path,exists
0,processed_base_dataset,data\processed\credit_default_base.csv,True
1,engineered_feature_dataset,data\processed\credit_default_features.csv,True
2,final_model_comparison,reports\tables\06_final_model_comparison.csv,True
3,best_model_metadata,reports\tables\06_best_model_metadata.csv,True
4,inference_sample,data\processed\inference_sample.csv,True
5,streamlit_model_artifact,models\best_credit_scoring_pipeline.joblib,True
